In [34]:
import spacy
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")  # Tải mô hình SpaCy cho Tokenization
stemmer = PorterStemmer()  # Stemmer cho Stemming
stop_words = set(stopwords.words('english'))  # Danh sách Stop-Words

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [35]:
doc = nlp("I love this movie and will watch it again")  # => tách câu thành câu thành các từ ( 1 token là một từ)
tokens = [token.text for token in doc]
print(tokens)

['I', 'love', 'this', 'movie', 'and', 'will', 'watch', 'it', 'again']


In [36]:
def preprocess_text(sentence):
    # Bước 1: Tokenization bằng SpaCy
    doc = nlp(sentence) # => tách câu thành câu thành các từ ( 1 token là một từ)
    tokens = [token.text for token in doc]

    # Giải thích Tokenization: Tokenization là quá trình chia văn bản thành các đơn vị nhỏ (token) như từ hoặc câu. Công dụng: Chuyển văn bản thô thành danh sách dễ xử lý, làm nền tảng cho các bước sau như Stemming hoặc Vocabulary-Matching. Nếu không có Tokenization, văn bản vẫn là chuỗi dài, khiến việc loại bỏ Stop-Words hoặc Stemming trở nên khó khăn, dẫn đến mô hình không thể phân tích từ riêng lẻ và độ chính xác giảm (ví dụ: "I love" bị coi là một từ duy nhất).

    # Bước 2: Loại bỏ Stop-Words
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]

    # Giải thích Stop-Words: Stop-Words là các từ phổ biến nhưng ít mang ý nghĩa như "the", "is", "and". Công dụng: Loại bỏ nhiễu để tập trung vào từ quan trọng, giảm kích thước dữ liệu và cải thiện tốc độ mô hình. Nếu không loại Stop-Words, từ vựng sẽ phồng to với các từ thừa, làm mô hình overfit (học nhiễu) hoặc chậm hơn, dẫn đến accuracy thấp hơn (ví dụ: "I love this movie" bị phân tích với "this" làm nhiễu).

    # Bước 3: Stemming để chuẩn hóa
    stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]

    # Giải thích Stemming: Stemming là quá trình cắt hậu tố để đưa từ về dạng gốc cơ bản (stem), như "running" → "run". Công dụng: Nhóm các biến thể từ (inflectional suffixes) để giảm kích thước từ vựng, giúp mô hình học hiệu quả hơn. Nếu không có Stemming, các từ như "love", "loved", "loving" được coi là riêng biệt, làm tăng kích thước vector TF-IDF và giảm khả năng tổng quát hóa mô hình (ví dụ: mô hình không nhận ra "loved" và "love" là cùng khái niệm).

    return stemmed_tokens

In [37]:
print(preprocess_text("I love this movie and will watch it again"))

['love', 'movi', 'watch']


In [38]:
from numpy import positive
def analyze_sentiment(sentence):
    processed_tokens = preprocess_text(sentence)

    # Danh sách từ khóa tích cực/tiêu cực (Vocabulary)
    positive_vocab = ["love", "great", "excel", "well", "happi"]
    negative_vocab = ["hate", "bad", "terribl", "sad", "bore"]

    # Vocabulary-Matching: Kỹ thuật khớp từ với danh sách từ vựng (vocabulary) để tìm mẫu hoặc ý nghĩa. Công dụng: Xác định từ khóa liên quan để phân loại hoặc trích xuất thông tin, giúp mô hình đơn giản nhưng hiệu quả cho dự án nhỏ. Nếu không có Vocabulary-Matching, bạn phải dựa hoàn toàn vào mô hình học máy phức tạp, dẫn đến over-engineering cho dự án đơn giản và tốn tài nguyên hơn (ví dụ: không khớp "great" với từ tích cực, làm phân loại sai).
    pos_count = sum(1 for token in processed_tokens if token in positive_vocab)
    neg_count = sum(1 for token in processed_tokens if token in negative_vocab)


    if pos_count > neg_count:
        return "Tích cực"
    elif neg_count > pos_count:
        return "Tiêu cực"
    else:
        return "Trung tính"

In [39]:
review = "The movie is better than the previous one"
result = analyze_sentiment(review)
print(f"Kết quả: {result}")  # Kết quả: Tích cực

Kết quả: Trung tính


In [40]:
stemmer.stem("unhappy")

'unhappi'

In [49]:
import spacy
import nltk
from nltk.corpus import stopwords

# Tải mô hình SpaCy và Stop-Words
nlp = spacy.load("en_core_web_sm")
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(sentence):
    # Bước 1: Tokenization bằng SpaCy
    doc = nlp(sentence) # => Tách câu thành các từ (1 token là một từ)
    tokens = [token for token in doc]

    # Giải thích Tokenization: Chia văn bản thành token (từ, dấu câu) để xử lý tiếp theo. Công dụng: Làm nền tảng cho Lemmatization, Stop-Words, và Vocabulary-Matching. Nếu không có, văn bản thô không thể phân tích từ riêng lẻ, dẫn đến lỗi khi khớp từ vựng hoặc giảm accuracy (ví dụ: "I love" bị coi là một từ).

    # Bước 2: Loại Stop-Words
    filtered_tokens = [token for token in tokens if token.text.lower() not in stop_words]

    # Giải thích Stop-Words: Loại từ thừa như "the", "is". Công dụng: Giảm nhiễu, tập trung vào từ mang ý nghĩa, giảm kích thước dữ liệu. Nếu không, từ thừa làm nhiễu Vocabulary-Matching, dẫn đến phân loại sai (ví dụ: "this" chiếm slot trong danh sách từ vựng).

    # Bước 3: Lemmatization bằng SpaCy
    lemmas = [token.lemma_ for token in filtered_tokens]

    # Giải thích Lemmatization: Chuyển từ về dạng gốc có nghĩa (như "better" → "good"). Công dụng: Nhóm biến thể từ, tăng độ chính xác khi khớp với từ vựng cảm xúc. Nếu dùng Stemming thay thế, "better" không chuyển thành "good", làm bỏ sót từ tích cực, giảm accuracy.

    return lemmas



def analyze_sentiment(sentence):
    processed_tokens = preprocess_text(sentence)

    # Danh sách từ khóa tích cực/tiêu cực (Vocabulary)
    positive_vocab = ["love", "great", "excel", "well", "happy"]
    negative_vocab = ["hate", "bad", "terribl", "sad", "bore"]


    # Vocabulary-Matching: Kỹ thuật khớp từ với danh sách từ vựng (vocabulary) để tìm mẫu hoặc ý nghĩa. Công dụng: Xác định từ khóa liên quan để phân loại hoặc trích xuất thông tin, giúp mô hình đơn giản nhưng hiệu quả cho dự án nhỏ. Nếu không có Vocabulary-Matching, bạn phải dựa hoàn toàn vào mô hình học máy phức tạp, dẫn đến over-engineering cho dự án đơn giản và tốn tài nguyên hơn (ví dụ: không khớp "great" với từ tích cực, làm phân loại sai).
    pos_count = sum(1 for token in processed_tokens if token in positive_vocab)
    neg_count = sum(1 for token in processed_tokens if token in negative_vocab)


    if pos_count > neg_count:
        return "Tích cực"
    elif neg_count > pos_count:
        return "Tiêu cực"
    else:
        return "Trung tính"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [50]:
review = "The movie is better than the previous one"
result = analyze_sentiment(review)
print(f"Review: {review}")
print(f"Kết quả: {result}")  # Kết quả: Tích cực

Review: The movie is better than the previous one
Kết quả: Tích cực
